# Colab tối ưu: OOXML + UniMERNet + Unlimited-OCR

**Vấn đề:** OCR cả trang ~57s × 5 = 286s, và T4 16GB không chứa nổi Unlimited-OCR 3B nếu load cùng UniMERNet.

**Cách tối ưu:**
1. `.docx` → extract OOXML (0.2s) — text, OMML, `*D.`, bảng. **Không OCR chữ.**
2. UniMERNet-tiny **batch** chỉ crop MathType WMF (16 ảnh nhỏ, không 5 trang).
3. Unlimited-OCR chỉ **hình vẽ** (`img_*`), mode `base`, `max_length=4096`. Load **sau khi unload** UniMERNet.
4. Auto-profile: T4 / A100 / CPU.

Runtime: **GPU T4**. Làm lại từ đầu: Disconnect and delete runtime → notebook mới → `Shift+Enter` từng cell. **Không** `pip install unimernet[full]` / `tokenizers` / `transformers==4.42.4`. Ô cài dùng `install_unimernet_colab()` (`--no-deps` + vá pytorch_utils).


## 1. GPU + clone code


In [ ]:
!nvidia-smi -L || echo "CPU only"
REPO_URL = "https://github.com/phuchoang2603/refurbished-marketplace.git"
REPO_BRANCH = "cursor/docx-to-azota-pipeline-4d56"
OUT_DIR = "/content/azota_out"
INJECT_LATEX = True   # [!m:$mathtype_N$] → $latex$ để Azota render


In [ ]:
import sys, shutil
from pathlib import Path
for p in ("/content/docx-to-azota", "/content/refurbished-marketplace", "/content/repo"):
    shutil.rmtree(p, ignore_errors=True)
!git clone -b {REPO_BRANCH} --depth 1 {REPO_URL} /content/refurbished-marketplace
shutil.copytree("/content/refurbished-marketplace/tools/docx-to-azota", "/content/docx-to-azota")
sys.path.insert(0, "/content/docx-to-azota")
print("OK")


In [ ]:
from convert import convert_docx, apply_unimernet_latex
from eval_timer import StepTimer
from colab_opt import (
    detect_profile, prepare_unimernet_checkpoint, free_cuda,
    vision_jobs_from_manifest, inject_latex_into_markup,
)
from vision import rasterize_formula_image, load_unimernet, unimernet_batch, load_unlimited_ocr, unlimited_ocr_one, strip_unlimited_ocr_det
from IPython.display import Image, display

PROFILE_NAME, PROFILE = detect_profile()
print("profile:", PROFILE_NAME)
print(PROFILE)
timer = StepTimer()


## 2. Cài ImageMagick + UniMERNet --no-deps (giữ transformers 5 của Colab)


In [ ]:
!pip -q install -U pillow pymupdf huggingface_hub
!apt-get -qq install -y imagemagick libmagickwand-dev >/dev/null
from install_colab import allow_wmf_in_imagemagick, install_unimernet_colab
allow_wmf_in_imagemagick()
install_unimernet_colab()
if PROFILE["ocr_figures"] or PROFILE["ocr_pages"]:
    !pip -q install -U einops addict easydict
print("deps OK")


## 3. Upload `.docx` hoặc `.pdf`


In [ ]:
from google.colab import files
uploaded = files.upload()
INPUT_PATH = "/content/" + next(iter(uploaded)) if uploaded else "/content/docx-to-azota/samples/de-vat-li-lan-3.docx"
SUFFIX = Path(INPUT_PATH).suffix.lower()
print(INPUT_PATH, SUFFIX)


## 4. Bước 1 — OOXML (thay pandoc / thay OCR chữ)


In [ ]:
import json
from pathlib import Path
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
MANIFEST = None
if SUFFIX != ".docx":
    raise SystemExit("Notebook tối ưu cho .docx. PDF: bật OCR trang trên A100 (colab_docx_to_azota.ipynb).")
with timer.step("Bước 1", "OOXML extract"):
    MANIFEST = convert_docx(INPUT_PATH, OUT_DIR)
print(json.dumps(MANIFEST["counts"], ensure_ascii=False, indent=2))
print("\n".join(Path(OUT_DIR, "markup.txt").read_text(encoding="utf-8").splitlines()[:25]))


## 5. Bước 2 — raster chỉ MathType WMF (không raster cả trang)


In [ ]:
png_dir = Path(OUT_DIR) / "sidecar_png"
png_dir.mkdir(exist_ok=True)
JOBS = []
with timer.step("Bước 2", "raster MathType WMF"):
    for aid, src in vision_jobs_from_manifest(MANIFEST, OUT_DIR, kinds=("mathtype",)):
        dest = png_dir / f"{aid}.png"
        got = rasterize_formula_image(src, dest, dpi=200)
        if got:
            JOBS.append((aid, got))
print(f"{len(JOBS)} công thức MathType")
if JOBS:
    display(Image(filename=str(JOBS[0][1])))


## 6. Bước 3 — UniMERNet-tiny batch (Universal MER)


In [ ]:
with timer.step("Bước 3a-load", f"download+load UniMERNet-{PROFILE['unimernet']}"):
    cfg = prepare_unimernet_checkpoint(PROFILE["unimernet"], "/content/models")
    um = load_unimernet(cfg_path=cfg, fp16=PROFILE["fp16"])
model, vis, device = um
print("device", device)


In [ ]:
PREDS = {}
if JOBS:
    with timer.step("Bước 3", f"UniMERNet batch {len(JOBS)} ảnh"):
        PREDS = unimernet_batch(model, vis, device, JOBS, batch_size=PROFILE["um_batch"])
    apply_unimernet_latex(MANIFEST, PREDS, Path(OUT_DIR))
    for k, v in list(PREDS.items())[:6]:
        print(k, "→", v[:120])
else:
    print("không có MathType")


In [ ]:
# Giải phóng VRAM trước khi OCR 3B
if PROFILE["unload_between"]:
    free_cuda(model, vis, um)
    um = model = vis = None
    print("đã unload UniMERNet")


## 7. Bước 3b — Unlimited-OCR chỉ hình vẽ (tùy GPU)

T4: `ocr_pages=False`, chỉ `img_*` (8 hình), mode **base**, `max_length=4096`.
A100: có thể OCR cả trang. **Không** OCR 5 full-page trên T4 (đó là lý do 286s / OOM).


In [ ]:
import torch
FIG_JOBS = vision_jobs_from_manifest(MANIFEST, OUT_DIR, kinds=("img",))
print("số hình vẽ:", len(FIG_JOBS))
OCR_MD = {}
if PROFILE["ocr_figures"] and FIG_JOBS:
    with timer.step("Bước 3b-load", "load Unlimited-OCR"):
        ocr_model, ocr_tok = load_unlimited_ocr()
    gundam = PROFILE["ocr_mode"] == "gundam"
    with timer.step("Bước 3b", f"OCR {len(FIG_JOBS)} hình"):
        for aid, src in FIG_JOBS:
            png = png_dir / f"{aid}.png"
            got = rasterize_formula_image(src, png, dpi=PROFILE["dpi"]) or src
            try:
                raw = unlimited_ocr_one(
                    ocr_model, ocr_tok, str(got),
                    f"{OUT_DIR}/ocr_raw/{aid}",
                    gundam=gundam,
                    max_length=PROFILE["max_ocr_len"],
                )
                OCR_MD[aid] = strip_unlimited_ocr_det(raw)
                print(aid, OCR_MD[aid][:180].replace("\n", " "))
            except torch.cuda.OutOfMemoryError:
                print("OOM tại", aid, "— dừng OCR hình, giữ [img:$…$]")
                free_cuda()
                break
    free_cuda(ocr_model, ocr_tok)
else:
    print("SKIP OCR hình (profile cpu hoặc ocr_figures=False)")


## 8. Bước 4 — ghép LaTeX vào markup Azota


In [ ]:
markup_path = Path(OUT_DIR) / "markup.txt"
text = markup_path.read_text(encoding="utf-8")
with timer.step("Bước 4", "ghép+fix LaTeX"):
    if INJECT_LATEX and PREDS:
        text = inject_latex_into_markup(text, PREDS)
        markup_path.write_text(text, encoding="utf-8")
timer.print_summary()
print("--- markup ---")
print("\n".join(text.splitlines()[:40]))


In [ ]:
from google.colab import files
!cd /content && zip -qr azota_out.zip azota_out
files.download("/content/azota_out.zip")
